[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shripada/ame5003-nlp/blob/main/primers/primer-1-nltk.ipynb)

**Click the badge above to open this lab in Google Colab.** Then choose *File → Save a copy in Drive* so your work is saved.

# Primer 1 — NLTK

**MSIS · AME 5003 and AME 5053 · Practice notebook · about 1 hour · not assessed**

*The parts of NLTK this course uses: corpora, tokenisers, stemmers, lemmatisers, taggers and
frequency counts.*

NLTK — the Natural Language Toolkit — is the library the labs reach for first. It was written
for teaching, which shows in two ways. It ships with the data as well as the code, so a corpus
is one line away, and it exposes each step of the pipeline separately, so you can look at the
output of one step before the next one runs.

This notebook is practice, not assessment. Nothing here is marked and no lab depends on it.
Work through it before lab 3 and the preprocessing steps will already be familiar.

**By the end you will be able to:**

1. Load a corpus that ships with NLTK, and read it as characters, words or sentences
2. Split text into sentences and words, and say why `split()` is not enough
3. Stem and lemmatise, and say which of the two you want
4. Tag parts of speech, and read the tags
5. Count with `FreqDist`, and find the words that carry the content

---
## Part 0 — Setup

Two cells. The first installs the library, the second downloads the data files NLTK keeps
separately from the code — word lists, the tokeniser tables, the tagger's model.

**Both run every session.** Colab wipes the machine between sessions, so a notebook that worked
yesterday starts from nothing today.

In [ ]:
!pip install -q nltk
print("Done.")

In [ ]:
import nltk

for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet",
            "averaged_perceptron_tagger_eng", "gutenberg", "movie_reviews"]:
    nltk.download(pkg, quiet=True)

print("NLTK data ready.")

> **Save your own copy now:** File → Save a copy in Drive. Otherwise your work goes when the
> tab does.

The download step is the one people forget. NLTK ships the code through pip and the data
through `nltk.download`, and a missing package raises `LookupError` at the moment you use it —
not at import, which is why it tends to surface halfway through a lab. The error message names
the package it wants; the fix is always to add that name to the cell above.

---
## Part 1 — Corpora: text that arrives with the library

NLTK ships around a hundred corpora. We use two of them in this course: `gutenberg`, which is a
handful of out-of-copyright books, and `movie_reviews`, which is 2,000 film reviews labelled
positive or negative and is the data lab 6 classifies.

Each corpus offers the same three views of its text, and which one you want depends on the task.

In [ ]:
from nltk.corpus import gutenberg

books = gutenberg.fileids()

print(len(books), "books:")
for name in books:
    print("  ", name)
# Verified output:
#   18 books:
#      austen-emma.txt
#      austen-persuasion.txt
#      austen-sense.txt
#      bible-kjv.txt
#      blake-poems.txt
#      bryant-stories.txt
#      burgess-busterbrown.txt
#      carroll-alice.txt
#      chesterton-ball.txt
#      chesterton-brown.txt
#      chesterton-thursday.txt
#      edgeworth-parents.txt
#      melville-moby_dick.txt
#      milton-paradise.txt
#      shakespeare-caesar.txt
#      shakespeare-hamlet.txt
#      shakespeare-macbeth.txt
#      whitman-leaves.txt

In [ ]:
emma_raw = gutenberg.raw("austen-emma.txt")        # one long string
emma_words = gutenberg.words("austen-emma.txt")    # a list of tokens
emma_sents = gutenberg.sents("austen-emma.txt")    # a list of lists of tokens

print("characters:", len(emma_raw))
print("tokens:    ", len(emma_words))
print("sentences: ", len(emma_sents))
print()
print("first 15 tokens:", emma_words[:15])
print()
print("sentence 4:", " ".join(emma_sents[3])[:90], "...")
# Verified output:
#   characters: 887071
#   tokens:     192427
#   sentences:  7752
#
#   first 15 tokens: ['[', 'Emma', 'by', 'Jane', 'Austen', '1816', ']', 'VOLUME', 'I', 'CHAPTER', 'I', 'Emma', 'Woodhouse', ',', 'handsome']
#
#   sentence 4: Emma Woodhouse , handsome , clever , and rich , with a comfortable home and happy disposit ...

`raw()` gives you the characters, and you would use it when you want to do your own splitting —
a regular expression over the text, say. `words()` and `sents()` give you the corpus already
tokenised, which is what you want when you are counting.

Two details worth noticing in that output. The punctuation is its own token, so the token count
is well above the number of words you would count by eye. And the first tokens include
`[`, `Emma`, `by`, `Jane`, `Austen`, `1816`, `]` — the header is part of the file, and a corpus
does not arrive clean.

### The types-and-tokens distinction, in two lines

A **token** is one occurrence of a word in the text; a **type** is one distinct word in the
vocabulary. *the cat sat on the mat* has six tokens and five types, because *the* occurs twice.
Session 3 uses this distinction constantly, so it is worth being able to compute both.

In [ ]:
words = [w.lower() for w in emma_words if w.isalpha()]

print("tokens:", len(words))
print("types: ", len(set(words)))
print(f"type/token ratio: {len(set(words)) / len(words):.3f}")
# Verified output:
#   tokens: 161600
#   types:  7079
#   type/token ratio: 0.044

`isalpha()` drops the punctuation and the numbers; lowercasing means *The* and *the* count as one
type. Both are decisions, not defaults — the ratio changes if you make them differently, which is
the point session 4 makes about normalisation.

### Exercise 1

Load `shakespeare-hamlet.txt` from the `gutenberg` corpus. Print its number of tokens, its number
of types, and its type/token ratio, using the same two filters as above.

Hamlet is about a fifth the length of *Emma*. Before you run it, guess whether its ratio will be
higher or lower — then read the note under the answer.

In [ ]:
# YOUR CODE HERE
# 1. hamlet = gutenberg.words("shakespeare-hamlet.txt")
# 2. keep the alphabetic tokens, lowercased
# 3. print len(), len(set()) and the ratio

Hamlet's ratio is the higher of the two, and the reason is mostly length rather than vocabulary.
The longer a text runs, the more often each new token is a word already seen, so the ratio falls
as the text grows. That makes the type/token ratio a poor way to compare texts of different
lengths — a fact worth carrying into any exercise that seems to be comparing two authors.

---
## Part 2 — Splitting text into sentences and words

`str.split()` splits on whitespace, and that is not tokenisation. Run this to see what it leaves
behind.

In [ ]:
text = "Mr. Rao paid Rs.5 for it. He didn't like it, really!"

print(text.split())
# Verified output:
#   ['Mr.', 'Rao', 'paid', 'Rs.5', 'for', 'it.', 'He', "didn't", 'like', 'it,', 'really!']

`it,` and `really!` keep their punctuation attached, so they will not match `it` and `really`
anywhere else in your data. `didn't` stays in one lump, hiding a negation that a sentiment
classifier needs to see. A tokeniser handles both.

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize

print(word_tokenize(text))
print()
for s in sent_tokenize(text):
    print("  ", repr(s))
# Verified output:
#   ['Mr.', 'Rao', 'paid', 'Rs.5', 'for', 'it', '.', 'He', 'did', "n't", 'like', 'it', ',', 'really', '!']
#
#      'Mr. Rao paid Rs.5 for it.'
#      "He didn't like it, really!"

`word_tokenize` made the comma and the exclamation mark their own tokens, and split `didn't` into
`did` and `n't` — pulling the negation out where it can be counted. `sent_tokenize` kept `Mr.`
and `Rs.5` intact and found the two real sentences, which splitting on `.` would not have done.

Both are statistical, not rule-based: `sent_tokenize` uses a model called Punkt, trained to tell
an abbreviation's full stop from a sentence's. That model is the `punkt_tab` download, which is
why the setup cell needs it.

### Regular-expression tokenisers, when you want a different rule

`word_tokenize` implements one particular set of conventions, and sometimes you want another.
`RegexpTokenizer` takes a pattern describing what a token *is* and keeps everything that matches.
Session 1 and session 2 are entirely about writing that pattern.

In [ ]:
from nltk.tokenize import RegexpTokenizer

tweet = "Reached #Manipal at 9:30am — cost Rs.450 :) @autorickshaw"

print("word_tokenize:", word_tokenize(tweet))
print()
print("words only:   ", RegexpTokenizer(r"\w+").tokenize(tweet))
print()
print("keep the tags:", RegexpTokenizer(r"[@#]\w+|\w+").tokenize(tweet))
# Verified output:
#   word_tokenize: ['Reached', '#', 'Manipal', 'at', '9:30am', '—', 'cost', 'Rs.450', ':', ')', '@', 'autorickshaw']
#
#   words only:    ['Reached', 'Manipal', 'at', '9', '30am', 'cost', 'Rs', '450', 'autorickshaw']
#
#   keep the tags: ['Reached', '#Manipal', 'at', '9', '30am', 'cost', 'Rs', '450', '@autorickshaw']

The third pattern keeps `#Manipal` and `@autorickshaw` whole, which the general-purpose tokeniser
splits at the symbol. Neither is wrong. What counts as a token depends on what you are going to
do with it, and for social media text the tags are often the most informative part.

### Exercise 2

Tokenise the sentence below three ways — with `split()`, with `word_tokenize`, and with
`RegexpTokenizer(r"\w+")` — and print the number of tokens each one produces.

In [ ]:
sentence = "Dr. Ambedkar's speech didn't end at 5 p.m.; it ran to 6:15."

# YOUR CODE HERE

Three tokenisers, three different answers to "how many words is this?". None of them is the
correct one; the question does not have an answer until a tokeniser has been chosen. That is the
whole of session 3's opening, and it is why a paper that reports a vocabulary size without saying
how it tokenised has told you very little.

---
## Part 3 — Stemming and lemmatisation

Both cut a word back to a root so that related forms count as one. They differ in what the root
is allowed to be. A **stem** is a shared key and need not be a word; a **lemma** is the
dictionary form and always is one. Session 5 is the lesson; here we are learning the calls.

In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer

porter = PorterStemmer()
wordnet = WordNetLemmatizer()

for word in ["studies", "studying", "organization", "was", "mice", "better"]:
    print(f"  {word:14} stem {porter.stem(word):10} lemma {wordnet.lemmatize(word)}")
# Verified output:
#     studies        stem studi      lemma study
#     studying       stem studi      lemma studying
#     organization   stem organ      lemma organization
#     was            stem wa         lemma wa
#     mice           stem mice       lemma mouse
#     better         stem better     lemma better

The stemmer is rules and nothing else, so it is fast, ignorant, and occasionally destructive:
`organization` becomes `organ`, and `was` becomes `wa`, which is not a word at all. That is
tolerable when the key is never shown to anyone — a search index, say.

The lemmatiser looks the word up, so `mice` reaches `mouse`. But it turned `was` into `wa` and
left `better` alone, which is no better than the stemmer managed — because it assumed both were
nouns. WordNet needs to be told the part of speech, as one letter: `n` noun, `v` verb,
`a` adjective, `r` adverb.

In [ ]:
for word, pos in [("was", "v"), ("better", "a"), ("studies", "v"), ("mice", "n")]:
    guessed = wordnet.lemmatize(word)
    told = wordnet.lemmatize(word, pos)
    print(f"  {word:10} no POS -> {guessed:8} told '{pos}' -> {told}")
# Verified output:
#     was        no POS -> wa       told 'v' -> be
#     better     no POS -> better   told 'a' -> good
#     studies    no POS -> study    told 'v' -> study
#     mice       no POS -> mouse    told 'n' -> mouse

Supplied with the part of speech, the lemmatiser is right every time; without it, it is wrong
about as often as the stemmer. So a lemmatiser needs a tagger in front of it, which is the next
part — and the reason session 6 comes before the labs that lemmatise.

---
## Part 4 — Part-of-speech tagging

`pos_tag` takes a list of tokens and returns a list of `(token, tag)` pairs. It expects tokens,
not a string, so it is always preceded by a tokeniser.

In [ ]:
from nltk import pos_tag

for sentence in ["We watched the play.", "They play cricket every Sunday."]:
    print(pos_tag(word_tokenize(sentence)))
# Verified output:
#   [('We', 'PRP'), ('watched', 'VBD'), ('the', 'DT'), ('play', 'NN'), ('.', '.')]
#   [('They', 'PRP'), ('play', 'VBP'), ('cricket', 'NN'), ('every', 'DT'), ('Sunday', 'NNP'), ('.', '.')]

In the first sentence `play` is tagged `NN`, a singular noun; in the second it is `VBP`, a
present-tense verb. Same string, different tags, and only the surrounding words decide — which is
the problem session 6 is about.

Those tags come from the Penn Treebank tagset, which has 45 of them and distinguishes `NN` from
`NNS` from `NNP`. When you only need the coarse class, ask for the universal tagset instead.

In [ ]:
print(pos_tag(word_tokenize("They play cricket every Sunday."), tagset="universal"))
# Verified output:
#   [('They', 'PRON'), ('play', 'VERB'), ('cricket', 'NOUN'), ('every', 'DET'), ('Sunday', 'NOUN'), ('.', '.')]

The tagger is right most of the time, not all of it. It is a model trained on newspaper text, and
it goes wrong where the evidence is thin.

In [ ]:
print(pos_tag(word_tokenize("I will book a flight.")))
# Verified output:
#   [('I', 'PRP'), ('will', 'MD'), ('book', 'NN'), ('a', 'DT'), ('flight', 'NN'), ('.', '.')]

`book` after `will` can only be a verb, and the tagger called it `NN`. Nothing warns you: a wrong
tag is returned exactly like a right one, and it then propagates into whatever you lemmatise or
count with it. Session 6 looks at why this particular kind of sentence is hard, and lab 3 has you
measure how often it happens.

### Exercise 3

Write a function that lemmatises a sentence properly: tag it first, map each Penn tag to the
letter WordNet wants, and lemmatise with that letter. Use the first letter of the Penn tag —
`J` for adjectives, `V` for verbs, `N` for nouns, `R` for adverbs — and fall back to `n` for
everything else, since that is what WordNet does anyway.

Test it on `"The mice were running and the children were happier than ever"`.

In [ ]:
sentence = "The mice were running and the children were happier than ever"

# YOUR CODE HERE
# 1. tokens = word_tokenize(sentence)
# 2. for token, tag in pos_tag(tokens):
# 3.     pos = {"J": "a", "V": "v", "N": "n", "R": "r"}.get(tag[0], "n")
# 4.     print the token, the tag, and wordnet.lemmatize(token, pos)

With the tags supplied, `were` reaches `be`, `running` reaches `run` and `happier` reaches
`happy`. Without them, all three survive untouched. Eight lines of glue, and the lemmatiser
starts doing what people assume it does out of the box.

spaCy does this in one call, because its tagger and lemmatiser were trained together and run in
one pipeline. That is primer 2, and it is a fair summary of the difference between the two
libraries: NLTK hands you the pieces, spaCy hands you the assembled thing.

---
## Part 5 — Counting: `FreqDist` and stop words

Counting is most of Unit II wearing different hats, so it is worth being fluent. NLTK's
`FreqDist` is a `collections.Counter` with a few extras — `most_common`, `hapaxes`, and a plot.

In [ ]:
from nltk import FreqDist

freq = FreqDist(words)          # the lowercased alphabetic tokens of Emma

print("distinct words:", len(freq))
print("total tokens:  ", freq.N())
print()
for word, n in freq.most_common(12):
    print(f"  {word:8} {n:6}")
# Verified output:
#   distinct words: 7079
#   total tokens:   161600
#
#     to         5239
#     the        5201
#     and        4896
#     of         4291
#     i          3178
#     a          3129
#     it         2528
#     her        2469
#     was        2398
#     she        2340
#     in         2188
#     not        2140

Every one of the top words is a function word. This is the observation session 4 starts from: the
most frequent words in any English text are the ones carrying the least information about what
the text is about. A **stop list** is the usual response — a list of words to drop before
counting.

In [ ]:
from nltk.corpus import stopwords

english_stops = set(stopwords.words("english"))

print("stop words in NLTK's English list:", len(english_stops))
print(sorted(english_stops)[:20])
print()
print("is 'not' on the list?", "not" in english_stops)
# Verified output:
#   stop words in NLTK's English list: 198
#   ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']
#
#   is 'not' on the list? True

In [ ]:
content = [w for w in words if w not in english_stops]

print(f"kept {len(content):,} of {len(words):,} tokens "
      f"({100 * len(content) / len(words):.0f}%)")
print()
for word, n in FreqDist(content).most_common(12):
    print(f"  {word:12} {n:5}")
# Verified output:
#   kept 73,149 of 161,600 tokens (45%)
#
#     mr            1153
#     emma           865
#     could          837
#     would          820
#     mrs            699
#     miss           599
#     must           567
#     harriet        506
#     much           486
#     said           484
#     one            452
#     weston         440

Now the list reads like a novel about a small group of people: names, `mr`, `miss`, `little`,
`think`. Removing about half the tokens made what remained far more informative — and lost
`not`, which was on the list. Session 4 and lab 3 both spend time on what that costs when the
task is sentiment rather than search.

### Exercise 4

Take the `movie_reviews` corpus, which is labelled `pos` and `neg`, and find the words that are
most characteristic of each label.

1. Get the tokens of each category with `movie_reviews.words(categories="pos")`
2. Lowercase, keep the alphabetic tokens, and drop the stop words
3. Print the twelve most common words in each

Then look at the two lists side by side. They will be more alike than you expect — which is the
observation lab 6 builds its classifier on top of.

In [ ]:
from nltk.corpus import movie_reviews

# YOUR CODE HERE

The two lists share most of their entries: `film`, `movie`, `one`, `like`. Frequency alone
barely separates the classes, because the commonest content words in film reviews are about films
regardless of the verdict. What separates them is which words are *relatively* more frequent in
one class than the other — the ratio, not the count. That is the idea behind Naïve Bayes in
session 15 and behind TF-IDF in session 18, and you have just seen why it is needed.

---
## Part 6 — Where NLTK stops

NLTK is a teaching library and it is honest about that. Two limits are worth knowing before you
choose it for something outside this course.

**It is slow.** Its taggers and parsers are pure Python. For a few thousand sentences that does
not matter; for a few million it does.

**The pieces are separate by design.** You saw that in exercise 3: tokenise, then tag, then map
the tag, then lemmatise, gluing four calls together yourself. That separation is exactly what
makes it good for teaching — you can inspect the output of every step — and exactly what makes it
tedious in production, where you want one call and a result.

spaCy takes the other position: one pipeline object, one call, everything computed at once. Which
is primer 2.

---
## What to remember

| you want | the call |
| --- | --- |
| a corpus | `gutenberg.words(fileid)`, `.sents(fileid)`, `.raw(fileid)` |
| sentences from a string | `sent_tokenize(text)` |
| tokens from a string | `word_tokenize(text)` |
| tokens on your own rule | `RegexpTokenizer(pattern).tokenize(text)` |
| a stem | `PorterStemmer().stem(word)` |
| a lemma | `WordNetLemmatizer().lemmatize(word, pos)` — supply the `pos` |
| tags | `pos_tag(tokens)`, or `pos_tag(tokens, tagset="universal")` |
| counts | `FreqDist(tokens).most_common(n)` |
| a stop list | `stopwords.words("english")` |

And the three things that will actually bite you: the download cell has to run every session; the
lemmatiser needs a part of speech or it assumes noun; and every tokeniser gives a different token
count, so the count is a property of your choice, not of the text.

**Next:** primer 2 does the same tour of spaCy, where all of this happens in one call.